# Shape-Based Screening

This notebook compares candidate molecular shapes against a supplied
3D reference conformation using RDKit.

## Accessing a 3D conformation

The input molecule must already contain a 3D conformation.
The function below retrieves that conformation without generating
or modifying atomic coordinates.

In [1]:
from rdkit import Chem


def get_3d_conformer(molecule, conformer_id=-1):
    """Retrieve an existing 3D conformer from an RDKit molecule."""
    if not isinstance(molecule, Chem.Mol):
        raise TypeError("Input must be an RDKit molecule.")

    if molecule.GetNumAtoms() == 0:
        raise ValueError("The molecule must contain atoms.")

    if molecule.GetNumConformers() == 0:
        raise ValueError("The molecule must contain a 3D conformation.")

    conformer = molecule.GetConformer(conformer_id)

    if not conformer.Is3D():
        raise ValueError("The selected conformer must be marked as 3D.")

    return conformer

## Loading the reference molecule

The reference molecule is loaded from an SDF file containing
a supplied 3D conformation.

If the file contains multiple molecular records, `record_index`
selects which record to load. Record indices start at zero.

Existing coordinates are preserved.

In [2]:
from pathlib import Path


def load_reference_3d(file_path, record_index=0):
    """Load a reference molecule with an existing 3D conformation."""
    path = Path(file_path)

    if not path.is_file():
        raise FileNotFoundError(f"SDF file not found: {path}")

    if (
        isinstance(record_index, bool)
        or not isinstance(record_index, int)
        or record_index < 0
    ):
        raise ValueError("Record index must be a non-negative integer.")

    supplier = Chem.SDMolSupplier(str(path), removeHs=False)

    if record_index >= len(supplier):
        raise IndexError("Record index is outside the SDF file.")

    molecule = supplier[record_index]

    if molecule is None:
        raise ValueError("The selected molecular record could not be read.")

    get_3d_conformer(molecule)

    return molecule

## Loading candidate molecules

Candidate molecules are loaded from an SDF file.

Each record must contain a 3D conformation. Existing coordinates
are preserved.

Each candidate receives a unique record identifier so that
screening results can be traced back to the input file.

In [3]:
def load_candidates_3d(file_path):
    """Load candidate molecules with existing 3D conformations."""
    path = Path(file_path)

    if not path.is_file():
        raise FileNotFoundError(f"SDF file not found: {path}")

    supplier = Chem.SDMolSupplier(str(path), removeHs=False)

    candidates = []

    for record_index, molecule in enumerate(supplier):
        if molecule is None:
            raise ValueError(
                f"Candidate record {record_index} could not be read."
            )

        get_3d_conformer(molecule)

        compound_id = f"record_{record_index}"

        candidates.append((compound_id, molecule))

    if not candidates:
        raise ValueError("The SDF file contains no candidate molecules.")

    return candidates

## Aligning molecular shapes

Each candidate is aligned to the supplied reference conformation
using molecular shape only.

Chemical feature matching is disabled. Copies of the input
molecules are used to preserve their original coordinates.

Alignment adjusts the candidate's position and orientation.
It does not generate a new internal conformation.

In [4]:
from rdkit.Chem import rdShapeAlign


def align_candidate_by_shape(
    reference,
    candidate,
    reference_conformer_id=-1,
    candidate_conformer_id=-1
):
    """Align a candidate to a reference using molecular shape only."""
    get_3d_conformer(reference, reference_conformer_id)
    get_3d_conformer(candidate, candidate_conformer_id)

    reference_copy = Chem.Mol(reference)
    aligned_candidate = Chem.Mol(candidate)

    rdShapeAlign.AlignMol(
        reference_copy,
        aligned_candidate,
        refConfId=reference_conformer_id,
        probeConfId=candidate_conformer_id,
        useColors=False
    )

    return reference_copy, aligned_candidate

## Calculating shape similarity

Shape similarity is calculated after alignment using RDKit's
grid-based molecular shape representation.

This provides an approximation of molecular volume overlap.
Shape Tanimoto distance is converted to similarity:

similarity = 1 - distance

Hydrogens are excluded from scoring. Grid spacing is expressed
in angstroms and controls the resolution of the approximation.

In [5]:
import math
from rdkit.Chem import rdShapeHelpers


def calculate_shape_similarity(
    reference,
    aligned_candidate,
    reference_conformer_id=-1,
    candidate_conformer_id=-1,
    grid_spacing=0.5
):
    """Calculate grid-based shape similarity after alignment."""
    get_3d_conformer(reference, reference_conformer_id)
    get_3d_conformer(aligned_candidate, candidate_conformer_id)

    if not math.isfinite(grid_spacing) or grid_spacing <= 0:
        raise ValueError("Grid spacing must be finite and positive.")

    distance = rdShapeHelpers.ShapeTanimotoDist(
        reference,
        aligned_candidate,
        confId1=reference_conformer_id,
        confId2=candidate_conformer_id,
        gridSpacing=grid_spacing,
        ignoreHs=True
    )

    return 1.0 - distance

## Screening and ranking candidates

Each candidate is aligned to the supplied reference conformation
and scored using grid-based shape Tanimoto similarity.

Candidates are supplied as (compound identifier, molecule) pairs.
Results are sorted by decreasing shape similarity.

Only the selected conformation of each molecule is compared.

In [6]:
def screen_by_shape(
    reference,
    candidates,
    reference_conformer_id=-1,
    candidate_conformer_id=-1,
    grid_spacing=0.5
):
    """Rank candidate molecules by similarity to a supplied 3D reference."""
    get_3d_conformer(reference, reference_conformer_id)

    results = []

    for compound_id, candidate in candidates:
        aligned_reference, aligned_candidate = align_candidate_by_shape(
            reference,
            candidate,
            reference_conformer_id=reference_conformer_id,
            candidate_conformer_id=candidate_conformer_id
        )

        similarity = calculate_shape_similarity(
            aligned_reference,
            aligned_candidate,
            reference_conformer_id=reference_conformer_id,
            candidate_conformer_id=candidate_conformer_id,
            grid_spacing=grid_spacing
        )

        results.append({
            "compound_id": compound_id,
            "shape_similarity": similarity
        })

    return sorted(
        results,
        key=lambda result: result["shape_similarity"],
        reverse=True
    )